# Job Recommendation — Collaborative Filtering for Flutter App

**Challenge:** [ENS Challenge Data #164](https://challengedata.ens.fr/challenges/164)

This notebook trains an ALS collaborative filtering model on job interaction sessions and exports:
1. A **trained model** (`model/als_model.npz`) — item factors + mappings for inference
2. A **jobs CSV** (`jobs_export.csv`) — structured job info ready for Flutter display
3. A **Flask/FastAPI inference snippet** — drop-in recommendation endpoint

**Workflow:**
```
User views jobs in Flutter app
       ↓
POST /recommend  {job_ids: [...], actions: [...]}
       ↓
ALS fold-in → top-10 job_ids
       ↓
Flutter looks up job details from local jobs_export.csv / SQLite
```

## 1. Imports & Configuration

In [ ]:
import ast, json, os, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Hyperparameters ────────────────────────────────────────────────────────────
WEIGHT_MAP     = {'apply': 5, 'view': 1}   # apply = stronger signal
ALS_FACTORS    = 64                         # latent dimensions
ALS_ITERATIONS = 25
ALS_REG        = 0.05
ALS_ALPHA      = 40    # confidence: C_ui = 1 + alpha * r_ui
TOP_K          = 10    # jobs returned per recommendation call

# ── Data paths ─────────────────────────────────────────────────────────────────
X_TRAIN_PATH   = 'x_train_Meacfjr.csv'
Y_TRAIN_PATH   = 'y_train_SwJNMSu.csv'
X_TEST_PATH    = 'x_test_jCBBNP2.csv'
JOBS_META_PATH = 'job_postings.json'   # optional — job metadata for CSV export

os.makedirs('model', exist_ok=True)
print('✓ Setup complete')

## 2. Load & Parse Session Data

In [ ]:
def load_sessions(path):
    df = pd.read_csv(path)
    df['job_ids'] = df['job_ids'].apply(ast.literal_eval)
    df['actions'] = df['actions'].apply(ast.literal_eval)
    return df

df_train = load_sessions(X_TRAIN_PATH)
df_y     = pd.read_csv(Y_TRAIN_PATH)
df_test  = load_sessions(X_TEST_PATH)

print(f'Train sessions : {len(df_train):,}')
print(f'Test  sessions : {len(df_test):,}')
print(f'Unique jobs    : {len({j for js in df_train["job_ids"] for j in js}):,}')
df_train.head(3)

## 3. Build Weighted Interaction Matrix

`apply` events contribute **5×** the signal weight of `view` events.

In [ ]:
sessions_list = df_train['session_id'].tolist()
session2idx   = {s: i for i, s in enumerate(sessions_list)}

all_jobs = sorted({j for js in df_train['job_ids'] for j in js})
job2idx  = {j: i for i, j in enumerate(all_jobs)}
idx2job  = {i: j for j, i in job2idx.items()}

rows, cols, data = [], [], []
for _, row in df_train.iterrows():
    s_i = session2idx[row['session_id']]
    for job, act in zip(row['job_ids'], row['actions']):
        j_i = job2idx.get(job)
        if j_i is not None:
            rows.append(s_i)
            cols.append(j_i)
            data.append(WEIGHT_MAP.get(act, 1))

R = sparse.csr_matrix(
    (data, (rows, cols)),
    shape=(len(session2idx), len(job2idx)),
    dtype=np.float32
)

print(f'Matrix shape : {R.shape}  (sessions × jobs)')
print(f'Non-zeros    : {R.nnz:,}')
print(f'Density      : {R.nnz / (R.shape[0]*R.shape[1]):.4%}')

## 4. ALS Training

Minimises weighted matrix factorisation for implicit feedback:

$$\min_{U,V} \sum_{u,i} c_{ui}(p_{ui} - \mathbf{u}_u^\top \mathbf{v}_i)^2 + \lambda(\|U\|_F^2 + \|V\|_F^2)$$

where $c_{ui} = 1 + \alpha r_{ui}$ and $p_{ui} = \mathbf{1}[r_{ui} > 0]$.

In [ ]:
def als_fit(R, factors=64, iterations=25, reg=0.05, alpha=40, verbose=True):
    """
    ALS for implicit feedback.
    Returns U (users × factors), V (items × factors).
    """
    n_users, n_items = R.shape

    C = R.copy().astype(np.float32)
    C.data = 1.0 + alpha * C.data          # confidence
    P = R.copy(); P.data[:] = 1.0          # preference

    U = (np.random.randn(n_users, factors) * 0.01).astype(np.float32)
    V = (np.random.randn(n_items, factors) * 0.01).astype(np.float32)
    reg_I = reg * np.eye(factors, dtype=np.float32)

    Rt, Ct, Pt = R.T.tocsr(), C.T.tocsr(), P.T.tocsr()

    for it in range(iterations):
        # Solve for U (fix V)
        VtV = V.T @ V
        for u in range(n_users):
            s, e = R.indptr[u], R.indptr[u+1]
            if s == e: continue
            ii, cv, pv = R.indices[s:e], C.data[s:e], P.data[s:e]
            Vi = V[ii]
            A = VtV + Vi.T @ (np.diag(cv - 1) @ Vi) + reg_I
            U[u] = np.linalg.solve(A, Vi.T @ (cv * pv))

        # Solve for V (fix U)
        UtU = U.T @ U
        for i in range(n_items):
            s, e = Rt.indptr[i], Rt.indptr[i+1]
            if s == e: continue
            uu, cv, pv = Rt.indices[s:e], Ct.data[s:e], Pt.data[s:e]
            Ui = U[uu]
            A = UtU + Ui.T @ (np.diag(cv - 1) @ Ui) + reg_I
            V[i] = np.linalg.solve(A, Ui.T @ (cv * pv))

        if verbose and (it + 1) % 5 == 0:
            print(f'  Iter {it+1}/{iterations}')

    return U, V


print('Training ALS...')
U, V = als_fit(R, factors=ALS_FACTORS, iterations=ALS_ITERATIONS,
               reg=ALS_REG, alpha=ALS_ALPHA)
print(f'Done.  U: {U.shape}  V: {V.shape}')

## 5. Offline Evaluation — Recall@K

In [ ]:
def recall_at_k(df_y, U, V, R, session2idx, idx2job, k=10):
    hits = 0
    for _, row in df_y.iterrows():
        s_i = session2idx.get(row['session_id'])
        if s_i is None: continue
        scores = V @ U[s_i]
        s, e = R.indptr[s_i], R.indptr[s_i+1]
        scores[R.indices[s:e]] = -np.inf  # mask seen
        top_k = {idx2job[i] for i in np.argsort(scores)[::-1][:k]}
        if row['job_id'] in top_k:
            hits += 1
    return hits / len(df_y), hits

print('Recall@K:')
for k in [1, 5, 10, 20]:
    r, h = recall_at_k(df_y, U, V, R, session2idx, idx2job, k=k)
    bar = '█' * int(r * 60)
    print(f'  @{k:>2}  {r:.4f}  {bar}')

## 6. Save Model for Inference

The Flutter backend only needs `V` (item factors) + the `job2idx`/`idx2job` mappings.
Test sessions are handled via **fold-in** (closed-form, no retraining).

In [ ]:
# Popularity fallback for cold-start (brand-new users with no history)
pop_scores = np.zeros(len(job2idx), dtype=np.float32)
for _, row in df_train.iterrows():
    for job, act in zip(row['job_ids'], row['actions']):
        j_i = job2idx.get(job)
        if j_i is not None:
            pop_scores[j_i] += WEIGHT_MAP.get(act, 1)

# Save item factors + popularity + job index
np.savez(
    'model/als_model.npz',
    V=V,
    pop_scores=pop_scores,
    job_ids=np.array(all_jobs),   # idx → job_id lookup
)

# Save job2idx mapping
pd.DataFrame({'job_id': all_jobs, 'idx': range(len(all_jobs))}).to_csv(
    'model/job_index.csv', index=False
)

print('✓ Saved  model/als_model.npz')
print('✓ Saved  model/job_index.csv')

## 7. Inference Function (Flutter Backend)

This is the **only function** your API needs to call at runtime.
Copy it into your Flask / FastAPI server.

In [ ]:
# ── Inference module — copy to your API server ─────────────────────────────────

def load_model(model_path='model/als_model.npz', index_path='model/job_index.csv'):
    """
    Load saved ALS artefacts. Call once at server startup.
    Returns: (V, pop_scores, job2idx, idx2job)
    """
    data     = np.load(model_path)
    V        = data['V']                    # (n_jobs, factors)
    pop_scores = data['pop_scores']         # (n_jobs,)
    df_idx   = pd.read_csv(index_path)
    job2idx  = dict(zip(df_idx['job_id'], df_idx['idx']))
    idx2job  = dict(zip(df_idx['idx'], df_idx['job_id']))
    return V, pop_scores, job2idx, idx2job


def recommend(
    job_ids:  list,          # ordered list of job IDs the user viewed/applied to
    actions:  list,          # matching list of 'view' | 'apply'
    V,
    pop_scores,
    job2idx,
    idx2job,
    top_k:    int  = 10,
    alpha:    float = 40.0,
    reg:      float = 0.05,
    weight_map: dict = {'apply': 5, 'view': 1},
) -> list:
    """
    Return top-K recommended job IDs for a session.

    Parameters
    ----------
    job_ids  : list of job IDs the user interacted with (oldest → newest)
    actions  : matching list of action strings ('view' or 'apply')
    ...

    Returns
    -------
    list of top_k job IDs (strings), excluding already-seen ones
    """
    n_jobs = len(job2idx)
    seen   = {job2idx[j] for j in job_ids if j in job2idx}

    # Cold-start: no known jobs → popularity ranking
    if not seen:
        scores = pop_scores.copy()
    else:
        # Build raw interaction vector
        r = np.zeros(n_jobs, dtype=np.float32)
        for job, act in zip(job_ids, actions):
            j_i = job2idx.get(job)
            if j_i is not None:
                r[j_i] += weight_map.get(act, 1)

        # ALS fold-in: u* = (V^T C V + λI)^{-1} V^T c·p
        c = 1.0 + alpha * r
        p = (r > 0).astype(np.float32)
        A = V.T @ (c[:, None] * V) + reg * np.eye(V.shape[1], dtype=np.float32)
        b = V.T @ (c * p)
        u_star = np.linalg.solve(A, b)
        scores = V @ u_star

    # Mask seen jobs
    scores[list(seen)] = -np.inf

    top_idx  = np.argsort(scores)[::-1][:top_k]
    return [str(idx2job[i]) for i in top_idx if i in idx2job]


# ── Quick smoke test ────────────────────────────────────────────────────────────
V_loaded, pop_loaded, j2i, i2j = load_model()

# Simulate a Flutter user who viewed two jobs
example_job_ids = list(df_train['job_ids'].iloc[0][:2])
example_actions = ['view', 'view']

recs = recommend(example_job_ids, example_actions, V_loaded, pop_loaded, j2i, i2j)
print('Example recommendations:', recs)

## 8. Export Jobs to CSV (for Flutter)

This section builds `jobs_export.csv` — one row per job with structured columns
that Flutter can consume directly (SQLite, Hive, or a REST endpoint).

**Expected job metadata format** (`job_postings.json`):
```json
{
  "JOB_ID": {
    "TITLE": "Ingénieur Système",
    "DESCRIPTION": "...",
    "SKILLS": [{"name": "Azure", "type": "hard"}, ...],
    "TASKS": [{"name": "Administrer des serveurs..."}],
    "LANGUAGES": [],
    "CERTIFICATIONS": [],
    "COURSES": []
  },
  ...
}
```

> If you don't have job metadata yet, run the **Job Extractor** section at the bottom.

In [ ]:
def extract_job_csv(job_postings: dict, output_path='jobs_export.csv') -> pd.DataFrame:
    """
    Flatten job metadata into a CSV with one row per job.
    
    Columns:
      job_id, title, description,
      skills_hard, skills_soft,        <- pipe-separated strings
      tasks,                            <- pipe-separated strings
      languages, certifications,        <- pipe-separated strings
      top_skill_1..top_skill_5          <- quick-access top skills
    """
    rows = []
    for job_id, meta in job_postings.items():
        # Parse skills
        raw_skills = meta.get('SKILLS', [])
        if isinstance(raw_skills, str):
            try: raw_skills = ast.literal_eval(raw_skills)
            except: raw_skills = []

        hard_skills = [
            s['name'] for s in raw_skills
            if isinstance(s, dict) and s.get('type') == 'hard'
        ]
        soft_skills = [
            s['name'] for s in raw_skills
            if isinstance(s, dict) and s.get('type') == 'soft'
        ]

        # Deduplicate preserving order, normalize to lowercase for display
        seen = set()
        deduped_hard = []
        for s in hard_skills:
            key = s.lower().strip()
            if key not in seen:
                seen.add(key)
                deduped_hard.append(s.strip())

        # Parse tasks
        raw_tasks = meta.get('TASKS', [])
        if isinstance(raw_tasks, str):
            try: raw_tasks = ast.literal_eval(raw_tasks)
            except: raw_tasks = []
        tasks = [t['name'] for t in raw_tasks if isinstance(t, dict) and 'name' in t]

        # Description — first 500 chars for card preview
        description = (meta.get('DESCRIPTION') or '').strip()
        description_preview = description[:500].rstrip()

        row = {
            'job_id':           str(job_id),
            'title':            (meta.get('TITLE') or '').strip(),
            'description':      description,
            'description_preview': description_preview,
            'skills_hard':      ' | '.join(deduped_hard),
            'skills_soft':      ' | '.join(soft_skills),
            'tasks':            ' | '.join(tasks),
            'languages':        ' | '.join(meta.get('LANGUAGES') or []),
            'certifications':   ' | '.join(meta.get('CERTIFICATIONS') or []),
            'n_skills':         len(deduped_hard),
            'n_tasks':          len(tasks),
        }
        # Top-5 skills as individual columns (handy for Flutter chip display)
        for k in range(1, 6):
            row[f'top_skill_{k}'] = deduped_hard[k-1] if len(deduped_hard) >= k else ''

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f'✓ Exported {len(df):,} jobs → {output_path}')
    return df


# ── Run if metadata file exists ─────────────────────────────────────────────────
if os.path.exists(JOBS_META_PATH):
    with open(JOBS_META_PATH, 'r', encoding='utf-8') as f:
        job_postings = json.load(f)
    df_jobs = extract_job_csv(job_postings)
    print(df_jobs.head(3).to_string())
else:
    print(f'[INFO] {JOBS_META_PATH} not found — skipping job metadata export.')
    print('       See section 9 to extract metadata from raw listings.')

## 9. Job Data Extractor

Use this if your job data lives in the challenge's raw format (one dict per listing, like the sample in the problem statement).  
Paste / load your raw listings and this will build the `job_postings.json` + `jobs_export.csv`.

In [ ]:
def parse_raw_listing(raw: str | dict) -> dict:
    """
    Parse a single job listing from the challenge format.
    Handles both plain-dict and stringified-list formats.

    Input example (from challenge):
        [
          'TITLE\\nIngénieur Système\\n...',
          'SECTION:\\nDESCRIPTION: ...',
          ...
        ]

    Returns a clean dict ready for extract_job_csv().
    """
    if isinstance(raw, str):
        try:
            raw = ast.literal_eval(raw)
        except Exception:
            raw = [raw]

    title, description, skills, tasks = '', '', [], []
    languages, certifications, courses = [], [], []

    current_section = None
    for block in raw:
        block = block.strip()

        # TITLE block
        if block.startswith('TITLE'):
            lines = block.split('\n')
            if len(lines) > 1:
                title = lines[1].strip()

        # SKILLS block
        elif block.startswith('SKILLS'):
            lines = block.split('\n')
            for line in lines[1:]:
                line = line.strip().lstrip('- ')
                if not line: continue
                try:
                    skill = ast.literal_eval(line)
                    if isinstance(skill, dict):
                        skills.append(skill)
                except Exception:
                    skills.append({'name': line, 'type': 'hard', 'value': None})

        # TASKS block
        elif block.startswith('TASKS'):
            lines = block.split('\n')
            for line in lines[1:]:
                line = line.strip().lstrip('- ')
                if not line: continue
                try:
                    task = ast.literal_eval(line)
                    if isinstance(task, dict):
                        tasks.append(task)
                except Exception:
                    tasks.append({'name': line, 'value': None})

        # LANGUAGES
        elif block.startswith('LANGUAGES'):
            lines = block.split('\n')[1:]
            languages = [l.strip().lstrip('- ') for l in lines if l.strip()]

        # CERTIFICATIONS
        elif block.startswith('CERTIFICATIONS'):
            lines = block.split('\n')[1:]
            certifications = [l.strip().lstrip('- ') for l in lines if l.strip()]

        # SECTION: DESCRIPTION blocks (summary / requirements)
        elif block.startswith('SECTION:'):
            desc_match = block.find('DESCRIPTION:')
            if desc_match != -1:
                snippet = block[desc_match + len('DESCRIPTION:'):].strip()
                description = (description + '\n' + snippet).strip()

    return {
        'TITLE':          title,
        'DESCRIPTION':    description,
        'SKILLS':         skills,
        'TASKS':          tasks,
        'LANGUAGES':      languages,
        'CERTIFICATIONS': certifications,
        'COURSES':        courses,
    }


# ── Demo: parse the sample listing from the problem statement ───────────────────
SAMPLE_LISTING = """['TITLE\nIngénieur Système\n\nSUMMARY\nNous recherchons un Ingénieur Système pour notre client.\n\nSECTION:\nDESCRIPTION: Nous recherchons un Ingénieur Système pour notre client. Le candidat idéal devra posséder des compétences variées en environnement système, virtualisation, Cloud et DevOps.\n\nSECTION:\nDESCRIPTION: Diplôme Bac +3/+5 en Informatique ou équivalent.\nMaîtrise des environnements systèmes Windows et Linux.\n\nSKILLS\n- {\'name\': \'Azure\', \'type\': \'hard\', \'value\': None}\n- {\'name\': \'DevOps\', \'type\': \'hard\', \'value\': None}\n- {\'name\': \'docker\', \'type\': \'hard\', \'value\': None}\n- {\'name\': \'kubernetes\', \'type\': \'hard\', \'value\': None}\n\nTASKS\n- {\'name\': \'administrer des serveurs middleware\', \'value\': None}\n- {\'name\': \'gérer des bases de données\', \'value\': None}']"""

parsed = parse_raw_listing(SAMPLE_LISTING)
print('Parsed title:  ', parsed['TITLE'])
print('Hard skills:   ', [s['name'] for s in parsed['SKILLS']])
print('Tasks:         ', [t['name'] for t in parsed['TASKS']])

In [ ]:
def build_jobs_from_raw_csv(raw_csv_path: str,
                             job_id_col: str = 'job_id',
                             content_col: str = 'content',
                             output_json: str = 'job_postings.json',
                             output_csv:  str = 'jobs_export.csv') -> pd.DataFrame:
    """
    Full pipeline: raw CSV → jobs_export.csv + job_postings.json
    
    Expected CSV columns:
      - job_id_col  : unique job identifier
      - content_col : raw listing string (the bracketed list format)
    """
    df_raw = pd.read_csv(raw_csv_path)
    postings = {}
    for _, row in df_raw.iterrows():
        jid  = str(row[job_id_col])
        meta = parse_raw_listing(row[content_col])
        postings[jid] = meta

    # Save intermediate JSON
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(postings, f, ensure_ascii=False, indent=2)
    print(f'✓ Saved {len(postings):,} job postings → {output_json}')

    # Build flat CSV
    return extract_job_csv(postings, output_csv)


# Usage:
# df_jobs = build_jobs_from_raw_csv('raw_job_listings.csv',
#                                     job_id_col='job_id',
#                                     content_col='listing_text')
print('build_jobs_from_raw_csv() ready to use — provide your raw listings CSV.')

## 10. Flask API Snippet

Drop this into a `server.py` alongside `model/`. Your Flutter app POSTs job history and gets 10 recommended job IDs back.

In [ ]:
FLASK_SNIPPET = '''
# server.py — Job Recommendation API
# pip install flask numpy pandas

from flask import Flask, request, jsonify
import numpy as np, pandas as pd

app = Flask(__name__)

# ── Load model once at startup ──────────────────────────────────────────────────
data       = np.load("model/als_model.npz")
V          = data["V"]                    # item latent factors
pop_scores = data["pop_scores"]           # popularity fallback
df_idx     = pd.read_csv("model/job_index.csv")
job2idx    = dict(zip(df_idx["job_id"].astype(str), df_idx["idx"]))
idx2job    = dict(zip(df_idx["idx"], df_idx["job_id"].astype(str)))

ALPHA, REG = 40.0, 0.05
WEIGHTS    = {"apply": 5, "view": 1}

# ── Recommend endpoint ──────────────────────────────────────────────────────────
@app.route("/recommend", methods=["POST"])
def recommend():
    body    = request.get_json(force=True)
    job_ids = body.get("job_ids", [])    # list[str]
    actions = body.get("actions", [])    # list["view"|"apply"]
    top_k   = int(body.get("top_k", 10))

    n_jobs = len(job2idx)
    seen   = {job2idx[j] for j in job_ids if j in job2idx}

    if not seen:
        scores = pop_scores.copy()
    else:
        r = np.zeros(n_jobs, dtype=np.float32)
        for j, a in zip(job_ids, actions):
            if j in job2idx:
                r[job2idx[j]] += WEIGHTS.get(a, 1)
        c = 1.0 + ALPHA * r
        p = (r > 0).astype(np.float32)
        A = V.T @ (c[:, None] * V) + REG * np.eye(V.shape[1], dtype=np.float32)
        u_star = np.linalg.solve(A, V.T @ (c * p))
        scores = V @ u_star

    scores[list(seen)] = -np.inf
    top_idx = np.argsort(scores)[::-1][:top_k]
    recs    = [str(idx2job[i]) for i in top_idx if i in idx2job]

    return jsonify({"recommended_job_ids": recs})

if __name__ == "__main__":
    app.run(port=5000)
'''

with open('model/server.py', 'w') as f:
    f.write(FLASK_SNIPPET.strip())

print('✓ Saved  model/server.py')
print()
print('Flutter POST example:')
print('''
  POST http://your-server:5000/recommend
  Content-Type: application/json

  {
    "job_ids": ["12345", "67890"],
    "actions": ["view", "apply"],
    "top_k": 10
  }

  → { "recommended_job_ids": ["11111", "22222", ...] }
''')

## 11. Flutter Integration Notes

```
jobs_export.csv  →  import into app
                     ├── SQLite via sqflite package
                     ├── Hive (key-value, offline)
                     └── Load at startup into a Map<String, Job>

Recommendation call:
  1. Track user interactions in session:  List<{jobId, action}>
  2. POST to /recommend  when user opens feed screen
  3. Receive [job_id_1, ..., job_id_10]
  4. Look up each ID in local job store → display cards
```

**CSV columns available in Flutter:**

| Column | Use |
|---|---|
| `job_id` | Primary key |
| `title` | Card headline |
| `description_preview` | 500-char card subtitle |
| `skills_hard` | Pipe-separated → split & show chips |
| `top_skill_1..5` | Pre-split top skills |
| `tasks` | Pipe-separated bullet points |
| `languages` / `certifications` | Filter tags |